# Grading research-math prompts

This grades the **prompt**, not the proof.

The rubric was frozen from six prompts in [`ShouqiaoW/erdos`](https://github.com/ShouqiaoW/erdos) —
researcher-written attempts at Erdős problems whose proofs were checked, several formalised in Lean.
Twelve axes, two subscales:

- **specification** (8 axes) — properties of the mathematical problem statement
- **procedure** (4 axes) — properties of the requested search and verification process

Each axis scores 0 / 1 / 2, so the total runs 0–24.

> **Not yet validated.** Nothing here has been checked against downstream output quality.
> Section 4 is the cheapest validation available; run it before trusting a number.

## 0. Setup

Needs `ANTHROPIC_API_KEY`. Set it in the shell before launching Jupyter
(`setx ANTHROPIC_API_KEY "sk-ant-..."`, then restart), or uncomment the line below.

In [ ]:
import os, sys

# os.environ["ANTHROPIC_API_KEY"] = "sk-ant-..."   # or set it in the shell

sys.path.insert(0, os.getcwd())
from mpe import grade, grade_corpus, compare, format_report, load_exemplars, render_rubric, DEFAULT_AXES

print("key set:", bool(os.environ.get("ANTHROPIC_API_KEY")))
print("exemplars:", [d.name for d in load_exemplars()])
print("axes:", len(DEFAULT_AXES))

In [ ]:
# The rubric being applied. Read this before reading any score.
print(render_rubric(DEFAULT_AXES))

## 1. Your prompt

Paste between the triple quotes, or point `PROMPT` at a file.

In [ ]:
PROMPT = r"""
Paste your prompt here.
"""

# from pathlib import Path
# PROMPT = Path("my_prompt.md").read_text(encoding="utf-8")

print(f"{len(PROMPT)} chars")

## 2. Grade it

In [ ]:
result = grade(PROMPT, name="my_prompt")
print(format_report(result))

## 3. Against the exemplars

Grades the six corpus prompts on the same axes and locates yours in that distribution.
Cached to `data/cache/exemplar_grades.json` after the first run — pass `refresh=True` to redo it.

n = 6 and one author, so read the percentile as a rough position, not a statistic.

In [ ]:
corpus_grades = grade_corpus()
for g in corpus_grades:
    print(f"{g.name:<16} {g.total:>3}/{g.max_total}")

comparison = compare(result, corpus_grades)
print()
print(format_report(result, comparison))

In [ ]:
# Where the corpus itself is strong and weak — i.e. which axes are house style
# rather than universal. An axis every exemplar maxes tells you little.
import statistics

for axis in DEFAULT_AXES:
    vals = [g.score_map().get(axis["id"], 0) for g in corpus_grades]
    spread = max(vals) - min(vals)
    print(f"{axis['name']:<32} mean {statistics.mean(vals):.2f}  spread {spread}  {vals}")

## 3b. Improve it

Proposes edits targeting the axes that scored below 2, applies the ones you accept,
and re-grades.

Edits are split by whether raising the axis requires **asserting mathematics**:

| class | axes | handling |
|---|---|---|
| structural | target claim, outcome left open, verification demand, search policy, stop condition, contamination control | applied by default |
| mathematical | formal setup, success criteria, ruled-out shortcuts, named pitfalls, permitted resources, reformulations | **always held for your review** |

The split is by axis, not by the model's self-report. A model asked to raise
`ruled_out_shortcuts` will happily invent plausible-sounding insufficient reductions —
which raises the score while making the prompt worse. Any gain that came from a
held-for-review edit is reported separately as *provisional* and does not count until
you have checked the claims.

In [ ]:
from mpe import improve, format_improvement

# accept="none" first: read what it wants to change before changing anything.
run = improve(PROMPT, accept="none")
print(format_improvement(run))

In [ ]:
# The claims you would be taking on trust if you accepted the held edits.
# Read these before deciding; this is the part the score cannot check for you.
for e in run.proposal.needing_review():
    print(f"[{e.id}] {e.axis}  ({e.rationale})")
    for c in e.claims_to_verify:
        print(f"    verify: {c}")
    print("    " + e.text.strip()[:400].replace("\n", "\n    "))
    print()

for row in run.proposal.cannot_fix:
    print(f"cannot fix without you — {row['axis']}: {row['why']}")

In [ ]:
from pathlib import Path

# Apply. "safe" = structural edits only (default); "all" = everything, flagged.
# To hand-pick after reading the proposal above:
#     improve(PROMPT, accept="ids", accept_ids=["e2", "e5"])
run = improve(PROMPT, accept="safe")
print(format_improvement(run))

# The revised text lives in run.revision.text and is lost when the kernel restarts,
# so write it out. The original is untouched.
if run.revision.applied:
    out = Path("my_prompt.revised.md")
    out.write_text(run.revision.text, encoding="utf-8")
    print(f"\nrevised prompt written to {out.resolve()}")

## 4. Validation — the degradation ladder

Take an exemplar, strip out one axis at a time, and re-grade. An axis whose removal
does not lower its own score is an axis the grader cannot see, and it should not be
counted in the total until it can be.

Three numbers come out:

| metric | question |
|---|---|
| sensitivity | did full ablation lower the targeted axis at all? |
| monotonicity | is full removal ≤ partial weakening ≤ original? |
| attribution | was the targeted axis the one that dropped most? |

**Cost.** One grading per ablation: `docs × axes × severities`, plus one baseline per doc.
The subset below is 1 doc × 4 axes × 2 severities = 8 ablations + 8 gradings + 1 baseline.
The full 6 × 12 × 2 grid is 144 ablations and 150 gradings — start small.
Everything caches under `data/cache/`.

In [ ]:
from mpe.ladder import run_ladder, format_ladder

docs = load_exemplars()[:1]
subset = [a for a in DEFAULT_AXES if a["id"] in {
    "ruled_out_shortcuts",   # the distinctive one — a rubric written into the prompt
    "success_criteria",
    "search_policy",
    "formal_setup",
}]

ladder = run_ladder(docs, axes=subset, severities=("partial", "full"))
print(format_ladder(ladder, axes=subset))

In [ ]:
# Every ablation, worst-behaved first: rows with target_drop == 0 are the failures.
for row in sorted(ladder.rows, key=lambda r: (r["target_drop"], r["axis"])):
    flag = " " if row["target_drop"] > 0 else "!"
    print(f"{flag} {row['axis']:<24}{row['severity']:<9}"
          f"{row['baseline_score']} -> {row['variant_score']}"
          f"   collateral {row['collateral']}")

## 5. Re-deriving the rubric

`DEFAULT_AXES` was frozen by hand from these six prompts. If you add exemplars from
other authors, re-induce and diff — an axis that survives across authors is a real
feature; one that does not is house style.

This is the fix for the single-author confound, and it matters more than any tuning below it.

In [ ]:
# from pathlib import Path
# from mpe import induce_axes
# from mpe.schema import save_axes
#
# induced = induce_axes(load_exemplars())
# for a in induced["axes"]:
#     print(f"{a['id']:<28}{a['subscale']:<16}{len(a['present_in'])}/6  {a['name']}")
# print("\nrejected:", induced["rejected"])
# save_axes(induced["axes"], Path("data/induced_axes.json"))

## What this does not tell you

The score measures resemblance to how six prompts by one author were built. It is not
evidence that a higher-scoring prompt produces better mathematics — that needs the
value-added measurement (same problem, your prompt vs. a rewrite, k generations each,
scored on partial progress), which is the next thing to build and the only one that
closes the loop.